### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for
ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple
schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures-

In [2]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A00C420BD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A00C57CE90>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [3]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="Name of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movie's rating out of 10")

In [4]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain': '1.3.9'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001A00C420BD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001A00C57CE90>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': 

In [5]:
# Regular output by the model
model.invoke("Provide details about the movie Transformers")

AIMessage(content="<think>\nOkay, the user wants details about the movie Transformers. Let me start by recalling what I know. Transformers is a 2007 movie directed by Michael Bay. It's part of the Transformers franchise, which started with the Hasbro toys and the 80s animated series. The main plot is about alien robots who can transform into vehicles or machines. The movie is a live-action sci-fi action film.\n\nFirst, I should mention the director, writers, and producers. Shia LaBeouf was the lead actor as Sam Witwicky. There's also Megan Fox as Mikaela Banning, and other main cast members like Josh Duhamel as Bumblebee. The Autobots and Decepticons are the two factions. The Decepticons are the villains led by Megatron, and the Autobots are led by Optimus Prime.\n\nThe movie was based on the 80s cartoons but had some new elements. It was a big success at the box office, breaking records. The visual effects were a major part, using CGI to create the Transformers. The soundtrack was by 

In [6]:
# Output after using pydantic
model_with_structure.invoke("Provide details about the movie Transformers")

Movie(title='Transformers', year=2007, director='Michael Bay', rating=6.7)

### Nested Structure

In [7]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie transformers")
response


MovieDetails(title='Transformers', year=2007, cast=[Actor(name='Shia LaBeouf', role='Sam Witwicky'), Actor(name='Megan Fox', role='Mikaela Banes'), Actor(name='Jericho', role='Bumblebee')], genres=['Action', 'Adventure', 'Sci-Fi'], budget=150.0)

### TypedDict
TypedDict provides a simpler alternative using Python's built-in typing, ideal when you dont need runtime validation.

In [8]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title:Annotated[str,...,"Name of the movie"]
    year:Annotated[int,...,"This year the movie was released"]
    director:Annotated[str,...,"The director of the movie"]
    rating:Annotated[float,...,"The movie's rating out of 10"]

model_withtypedict = model.with_structured_output(MovieDict)
response = model_withtypedict.invoke("Please provide the details of the movie Avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [9]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man'},
  {'name': 'Chris Evans', 'role': 'Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Hawkeye'}],
 'genres': ['Action', 'Adventure', 'Science Fiction'],
 'title': 'Avengers',
 'year': 2012}

In [10]:
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

### Data Classes
A data class is a class typically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorator

In [ ]:
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")

In [ ]:
# USing Pydantic
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact info  of a person"""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages":[{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])



name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [16]:
# Using Typedict
# USing Pydantic
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact info  of a person"""
    name: str # "The name of the person"
    email: str # "The email address of the person"
    phone: str # "The phone number of the person"

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo
)

result = agent.invoke({
    "messages":[{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result["structured_response"])



name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [18]:
# Using DataClass
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass 
class ContacInfo:
    """Contact info of a person"""
    name: str
    email: str 
    phone: str 

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo # Auto Selects provider strategy 
)

result = agent.invoke({
    "messages":[{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]


ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')